## Pipeline Audit Log — Failure Monitoring Dashboard

This notebook provides a **real-time view of pipeline execution health** across our medallion architecture (Bronze → Silver → Gold). It queries audit log tables to surface:

- **Overall failure rates** across all pipeline layers
- **Failure root-cause analysis** with error types and detailed messages
- **Job-level failure breakdown** to identify the most problematic pipelines
- **Alerting trail** showing which failures triggered notifications

**Data Sources:**
| Layer | Table | Description |
|-------|-------|-------------|
| Bronze | `bronze.logs.audit_logs` | Raw ingestion audit events |
| Gold | `hgi.gold.audit_logs` | Curated, enriched audit events |

In [0]:
%sql
-- Combined audit log view across all layers
CREATE OR REPLACE TEMP VIEW v_all_audit_logs AS
SELECT *, 'bronze' AS source_layer FROM bronze.logs.audit_logs
UNION ALL
SELECT *, 'gold' AS source_layer FROM hgi.gold.audit_logs;

SELECT
  job_name,
  customer_id,
  status,
  alert_type,
  error_type,
  layer,
  rows_processed,
  ROUND(duration_ms / 1000.0, 2) AS duration_sec,
  timestamp,
  run_date,
  source_layer
FROM v_all_audit_logs
ORDER BY timestamp DESC;

In [0]:
%sql
-- Failures by layer and date
SELECT
  run_date,
  source_layer,
  COUNT(*) AS failure_count,
  COLLECT_SET(error_type) AS error_types
FROM v_all_audit_logs
WHERE status = 'failure'
GROUP BY run_date, source_layer
ORDER BY run_date DESC, source_layer;

In [0]:
%sql
-- Alert notification trail for failures
SELECT
  timestamp,
  job_name,
  alert_type,
  email_notified,
  error_type,
  LEFT(error_reason, 150) AS error_summary
FROM v_all_audit_logs
WHERE status = 'failure'
  AND alert_type LIKE '%CRITICAL%'
ORDER BY timestamp DESC;